In [0]:
STORAGE_ACCOUNT  = "storageaccountfull"
BRONZE_CONTAINER = "bronze"
BRONZE_PATH      = f"abfss://{BRONZE_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net"

CATALOG          = "e-commerce"
SILVER_DB        = "silver"
TARGET_TABLE     = "dim_products"
NATURAL_KEY      = "product_id"

In [0]:
from pyspark.sql.functions import (
    col, trim, initcap, when, lit,
    current_timestamp, expr,
    row_number, monotonically_increasing_id
)
from pyspark.sql.types import IntegerType
from pyspark.sql.window import Window
from delta.tables import DeltaTable

spark.sql(f"USE CATALOG `{CATALOG}`")
spark.sql(f"CREATE DATABASE IF NOT EXISTS {SILVER_DB}")
print("Imports ready | Database ensured")

In [0]:
df_bronze = (
    spark.read
    .format("parquet")
    .load(f"{BRONZE_PATH}/products")
    .withColumn("source_file", col("_metadata.file_path"))
)

print(f"Bronze rows : {df_bronze.count():,}")
df_bronze.printSchema()
df_bronze.show(5, truncate=False)

In [0]:
df_clean = (
    df_bronze
    .filter(col(NATURAL_KEY).isNotNull())
    .withColumn("product_id",   trim(col("product_id")))
    .withColumn("product_name", trim(col("product_name")))
    .withColumn("brand",        trim(initcap(col("brand"))))
    .withColumn("category",     trim(initcap(col("category"))))
    .withColumn("cocoa_percent", col("cocoa_percent").cast(IntegerType()))
    .withColumn("weight_g",      col("weight_g").cast(IntegerType()))
    .withColumn("cocoa_percent",
        when((col("cocoa_percent") < 0) | (col("cocoa_percent") > 100), lit(None))
       .otherwise(col("cocoa_percent"))
    )
    .withColumn("weight_g",
        when(col("weight_g") <= 0, lit(None))
       .otherwise(col("weight_g"))
    )
    .withColumn("cocoa_tier",
        when(col("cocoa_percent") < 30,  lit("White"))
       .when(col("cocoa_percent") < 50,  lit("Milk"))
       .when(col("cocoa_percent") < 70,  lit("Semi-Dark"))
       .otherwise(lit("Dark"))
    )
    .withColumn("weight_band",
        when(col("weight_g") <= 50,  lit("Small (≤50g)"))
       .when(col("weight_g") <= 100, lit("Medium (51-100g)"))
       .otherwise(lit("Large (>100g)"))
    )
)

print(f"Clean rows : {df_clean.count():,}")

In [0]:
before     = df_clean.count()
df_deduped = df_clean.dropDuplicates([NATURAL_KEY])
print(f"Before dedup : {before:,}  |  After : {df_deduped.count():,}")

In [0]:
import warnings
warnings.filterwarnings("ignore", message=".*No Partition Defined for Window operation.*")

if spark.catalog.tableExists(f"{SILVER_DB}.{TARGET_TABLE}"):
    existing_sk = spark.table(f"{SILVER_DB}.{TARGET_TABLE}").select("product_sk", NATURAL_KEY)
    max_sk = existing_sk.agg({"product_sk": "max"}).collect()[0][0] or 0

    df_with_sk = df_deduped.join(existing_sk, on=NATURAL_KEY, how="left")
    df_existing = df_with_sk.filter(col("product_sk").isNotNull())
    df_new = (
        df_with_sk.filter(col("product_sk").isNull()).drop("product_sk")
        .withColumn("_row_id", monotonically_increasing_id())
    )
    window_new = Window.orderBy("_row_id")
    df_new = (
        df_new
        .withColumn("product_sk", (row_number().over(window_new) + max_sk).cast(IntegerType()))
        .drop("_row_id")
    )
    df_with_sk = df_existing.unionByName(df_new)
    print(f"Existing products (SK reused) : {df_existing.count():,}")
    print(f"New products (SK assigned)    : {df_new.count():,}")

else:
    window_new = Window.orderBy(NATURAL_KEY)
    df_with_sk = (
        df_deduped
        .withColumn("product_sk", row_number().over(window_new).cast(IntegerType()))
    )
    print(f"First run — assigning SKs 1 -> {df_with_sk.count():,}")

In [0]:
df_final = (
    df_with_sk
    .withColumn("ingestion_time", current_timestamp())
    .withColumn("updated_at",     current_timestamp())
)

In [0]:
df_dim = df_final.select(
    col("product_sk"),
    col("product_id"),
    col("product_name"),
    col("brand"),
    col("category"),
    col("cocoa_percent"),
    col("cocoa_tier"),
    col("weight_g"),
    col("weight_band"),
    col("ingestion_time"),
    col("source_file"),
    col("updated_at"),
)

print("Final dim schema:")
df_dim.printSchema()
df_dim.show(5, truncate=False)

In [0]:
if not spark.catalog.tableExists(f"{SILVER_DB}.{TARGET_TABLE}"):
    df_dim.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(f"{SILVER_DB}.{TARGET_TABLE}")
    print(f"{SILVER_DB}.{TARGET_TABLE} created")

else:
    df_dim.createOrReplaceTempView("dim_products_updates")

    spark.sql(f"""
        MERGE INTO {SILVER_DB}.{TARGET_TABLE}  AS target
        USING dim_products_updates             AS source
        ON target.product_id = source.product_id

        WHEN MATCHED THEN UPDATE SET
            target.product_name   = source.product_name,
            target.brand          = source.brand,
            target.category       = source.category,
            target.cocoa_percent  = source.cocoa_percent,
            target.cocoa_tier     = source.cocoa_tier,
            target.weight_g       = source.weight_g,
            target.weight_band    = source.weight_band,
            target.updated_at     = source.updated_at

        WHEN NOT MATCHED THEN INSERT *
    """)
    print(f"MERGE complete -> {SILVER_DB}.{TARGET_TABLE}")

In [0]:
spark.sql(f"SELECT COUNT(*) AS total_rows FROM {SILVER_DB}.{TARGET_TABLE}").show()

spark.sql(f"""
    SELECT brand, category, cocoa_tier, COUNT(*) AS product_count
    FROM {SILVER_DB}.{TARGET_TABLE}
    GROUP BY brand, category, cocoa_tier
    ORDER BY brand, category
""").show(30)

spark.sql(f"SELECT * FROM {SILVER_DB}.{TARGET_TABLE} LIMIT 5").show(truncate=False)

In [0]:
spark.sql(f"SELECT COUNT(*) AS total_rows FROM {SILVER_DB}.{TARGET_TABLE}").show()